In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import math
import dask
import dask.dataframe as dd
from dask.distributed import Client
from dask.distributed import LocalCluster

/home/aelyoussoufi/miniconda3/envs/latest/lib/python3.10/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
df = dd.read_csv('combined_snow_water.csv')
#df['Value'] = df['Value'].replace(55537, np.nan)
#max_value = df['Value'].max().compute()
#print(max_value)

In [ ]:

# Read and process the SWE data
df = dd.read_csv('combined_snow_water.csv')
df['Value'] = df['Value'] / 1000  # Scale SWE values to meters

# Filter out unrealistically high SWE values
df = df.assign(Value=df['Value'].where(df['Value'] < 32.767, np.nan))

# Compute the data to use with GeoPandas
df = df.compute()

# Create geometry for GeoDataFrame
geometry = [Point(xy) for xy in zip(df['X'], df['Y'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry)

# Set the CRS (Coordinate Reference System)
gdf.set_crs(epsg=4326, inplace=True)  # Adjust EPSG code if necessary

# Load and process the shapefile for U.S. states
us_states = gpd.read_file("cb_2018_us_state_20m.shp")
contiguous_us_states = us_states[~us_states['STUSPS'].isin(['AK'])]  # Exclude Alaska

# Plot the data
fig, ax = plt.subplots(figsize=(12, 10))

# Plot state boundaries
contiguous_us_states.boundary.plot(ax=ax, linewidth=0.8, color='black')
contiguous_us_states.plot(ax=ax, color='lightgrey', edgecolor='black', linewidth=0.8)

# Plot SWE points
gdf.plot(column='Value', ax=ax, markersize=20, cmap='rainbow', legend=True)

# Customize plot limits and labels
plt.xlim(-125, -66)
plt.ylim(20, 60)
plt.title("Snow Water Equivalent (SWE) Distribution in the U.S.")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

# Save and show the plot
plt.savefig("SWEDist.png", facecolor="white")
plt.show()

In [ ]:
# Define the latitude bins and calculate mean SWE for each bin
latitude_bins = np.arange(20, 51, 1)  # 1-degree latitude bins from 20° to 50° latitude
gdf['Latitude_bin'] = pd.cut(gdf['geometry'].y, bins=latitude_bins)  # Bin latitudes

#mean SWE per latitude bin
latitude_swe = gdf.groupby('Latitude_bin')['Value'].mean().reset_index()
latitude_swe['Latitude_center'] = latitude_swe['Latitude_bin'].apply(lambda x: x.mid)  # Get the center of each bin

#heatmap
plt.figure(figsize=(10, 6))
plt.scatter(latitude_swe['Latitude_center'], latitude_swe['Value'], 
            c=latitude_swe['Value'], cmap='rainbow', s=100, edgecolor='black')

cbar = plt.colorbar()
cbar.set_label("Average SWE (mm)", rotation=270, labelpad=15)
plt.xlabel("Latitude")
plt.ylabel("Average SWE (mm)")
plt.title("Heatmap of Average Snow Water Equivalent by Latitude")
plt.grid(True)

plt.show()

In [ ]:
from scipy.interpolate import griddata


latitudes = gdf.geometry.y
longitudes = gdf.geometry.x
swe_values = gdf['Value']

# Remove any NaN values
valid_points = ~np.isnan(swe_values)  # Mask for non-NaN values
latitudes = latitudes[valid_points]
longitudes = longitudes[valid_points]
swe_values = swe_values[valid_points]


grid_lat, grid_lon = np.mgrid[20:50:200j, -125:-65:200j]  # Adjust grid resolution as needed

# Ignore NaNs
grid_swe = griddata((latitudes, longitudes), swe_values, (grid_lat, grid_lon), method='cubic')

# Plot the contour
plt.figure(figsize=(12, 10))
contour = plt.contourf(grid_lon, grid_lat, grid_swe, levels=20, cmap='cool')  # Adjust levels as needed
plt.colorbar(contour, label='Snow Water Equivalent (SWE) in mm')

# Overlay U.S. state boundaries for context
contiguous_us_states.boundary.plot(ax=plt.gca(), linewidth=0.8, color='black')

# Customizing the plot
plt.title("Contour Plot of Snow Water Equivalent (SWE) in the U.S.")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(linestyle='--', alpha=0.5)
plt.show()


In [ ]:
geometry = [Point(xy) for xy in zip(df['X'], df['Y'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry)


gdf.set_crs(epsg=4326, inplace=True)  # Adjust EPSG code if necessary


us_states = gpd.read_file("/Users/aelyoussouri/Documents/Research/cb_2018_us_state_20m/cb_2018_us_state_20m.shp")
contiguous_us_states = us_states[~us_states['STUSPS'].isin(['AK'])]  # Exclude Alaska


fig, ax = plt.subplots(figsize=(12, 10))


contiguous_us_states.boundary.plot(ax=ax, linewidth=0.8, color='black')
contiguous_us_states.plot(ax=ax, color='lightgrey', edgecolor='black', linewidth=0.8)


gdf.plot(column='Value', ax=ax, markersize=20, cmap='cool', legend=True)




plt.xlim(-125, -105)  
plt.ylim(30, 50)      

plt.title("Snow Water Equivalent (SWE) Distribution in U.S. Mountain Regions")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.savefig("SWEDist_MountainRegions.png", facecolor="white")
plt.show()

## 